# ML-07 — Baseline Action Score and Top-20 Review

## Lane: Refresh / Content Opportunity Scoring

This notebook builds a transparent hand-written baseline for prioritizing content pages for human review.

The rule follows the ML-06 signal audit:

- CTR relative to position was **CONFIRMED** and is the main weakness signal.
- Visibility was **MIXED** but remains useful as an impact signal.
- Position volatility was **MIXED** and is used only as supporting context.
- Average position is used for fair CTR comparison and eligibility filtering.

The baseline is a decision-support ranking. It does not automatically edit content, prove a search-engine ranking factor, or claim that a refresh would cause recovery.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy

In [40]:
import os
import json
import subprocess
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

In [18]:
REPO_URL = "https://github.com/Zafar488/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

# Robust repository setup
if Path.cwd().name == REPO_DIR:
    repo_root = Path.cwd()
else:
    repo_root = Path("/content") / REPO_DIR

    if not repo_root.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URL,
                str(repo_root),
            ],
            check=True,
        )

    os.chdir(repo_root)

print("Working directory:", Path.cwd())

# Load Hugging Face token safely
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN missing. Add it in the Colab Secrets panel."
    )

user = whoami(token=HF_TOKEN)

print(
    "Hugging Face login successful:",
    user["name"],
)

# DuckDB connection
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "baseline_action_score.csv"
METRICS_PATH = OUTPUT_DIR / "baseline_metrics.json"

print("DuckDB connection ready.")
print("Development month: March 2026")
print("Feature window: March 1–15")
print("Outcome window: March 16–31 — signal evaluation only")

Working directory: /content/flyrank-ml-internship
Hugging Face login successful: zafar4050
DuckDB connection ready.
Development month: March 2026
Feature window: March 1–15
Outcome window: March 16–31 — signal evaluation only


## Baseline Data Frame

One row represents one anonymized content page for one anonymized client.

The feature window is March 1–15, 2026.

The outcome window is loaded only to repeat the two required signal checks. Outcome measurements are never used in:

- baseline eligibility;
- score components;
- reason codes;
- action labels;
- confidence labels;
- exported ranked queue.

The baseline score uses only measurements available during the feature window.

In [19]:
page_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature-window impressions
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feature_impressions,

        -- Feature-window clicks
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feature_clicks,

        -- Feature-window average position
        AVG(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_avg_position,

        -- Feature-window position volatility
        STDDEV_SAMP(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_position_volatility,

        -- Feature-window available days
        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS feature_available_days,

        -- Outcome-window impressions: audit only
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS outcome_impressions,

        -- Outcome-window available days: audit only
        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS outcome_available_days

    FROM {MARCH_FACT}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Raw page rows:", f"{len(page_frame):,}")

page_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw page rows: 331,437


,client_hash_id,content_hash_id,feature_impressions,feature_clicks,feature_avg_position,feature_position_volatility,feature_available_days,outcome_impressions,outcome_available_days
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,5.222776,2.612218,13,70.0,16
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,1.0,4.638889,2.521436,9,8.0,7
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,3.737399,2.554586,15,680.0,16
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,3.597222,2.900551,9,14.0,8
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,6.156643,1.144948,14,1614.0,16


#Engineer Feature-Window Signals — Code Cell

In [20]:
# Feature-window CTR
page_frame["feature_ctr"] = np.where(
    page_frame["feature_impressions"] > 0,
    page_frame["feature_clicks"]
    / page_frame["feature_impressions"],
    np.nan,
)

# Fill volatility where too few valid observations existed
volatility_fill = (
    page_frame["feature_position_volatility"].median()
)

page_frame["feature_position_volatility"] = (
    page_frame["feature_position_volatility"]
    .fillna(volatility_fill)
)

print("Feature-window signals created.")
print("Volatility fill value:", round(float(volatility_fill), 4))

Feature-window signals created.
Volatility fill value: 4.4128


#Audit Slice — Code Cell

In [21]:
# This slice may use outcome availability because it is used only
# for checking whether the two candidate signals hold.
audit_frame = page_frame[
    (page_frame["feature_impressions"] >= 100)
    & (page_frame["feature_available_days"] >= 5)
    & (page_frame["outcome_available_days"] >= 5)
].copy()

audit_frame["feature_daily_impressions"] = (
    audit_frame["feature_impressions"]
    / audit_frame["feature_available_days"]
)

audit_frame["outcome_daily_impressions"] = (
    audit_frame["outcome_impressions"]
    / audit_frame["outcome_available_days"]
)

# Audit-only future-window proxy
audit_frame["is_declining_proxy"] = (
    audit_frame["outcome_daily_impressions"]
    < 0.80 * audit_frame["feature_daily_impressions"]
).astype(int)

print("Audit rows:", f"{len(audit_frame):,}")
print(
    "Audit-only declining proxy rate:",
    round(audit_frame["is_declining_proxy"].mean(), 3),
)

Audit rows: 76,201
Audit-only declining proxy rate: 0.345


#Baseline Slice — Code Cell

In [22]:
# This slice uses only feature-window information.
# No outcome-window condition is used.
baseline_frame = page_frame[
    (page_frame["feature_impressions"] >= 500)
    & (page_frame["feature_available_days"] >= 5)
    & (page_frame["feature_avg_position"] > 0)
    & (page_frame["feature_avg_position"] <= 20)
    & (page_frame["feature_ctr"].notna())
].copy()

print(
    "Feature-only baseline population:",
    f"{len(baseline_frame):,}",
)

assert len(baseline_frame) > 0
assert (baseline_frame["feature_impressions"] >= 500).all()
assert (baseline_frame["feature_available_days"] >= 5).all()
assert (baseline_frame["feature_avg_position"] > 0).all()
assert (baseline_frame["feature_avg_position"] <= 20).all()

print("Feature-only eligibility check passed.")

Feature-only baseline population: 34,078
Feature-only eligibility check passed.


# 1. My Rule and Its Reason Code

Before encoding the rule, I repeat two signal checks.

## Signal Check 1 — Visibility

Visibility is linked to review priority because a high-exposure page has more potential business impact.

ML-06 found a **MIXED** relationship with later decline. Therefore, visibility is used only as an impact weight, not as proof of weakness.

## Signal Check 2 — CTR Relative to Position

CTR relative to position is linked to FlyRank's CTR-review logic.

ML-06 found this signal **CONFIRMED**. Therefore, below-expected CTR is the main weakness signal.

## Plain-Words Rule

A page enters the ranked queue only when:

1. it received at least 500 impressions during the feature window;
2. its average position was between 1 and 20;
3. its CTR was below the median CTR of pages in the same position band.

Pages are ranked higher when:

- their CTR is further below the position-band median;
- their visibility is higher;
- their position was more volatile.

## Score

`baseline_action_score =`

- 60% position-adjusted CTR weakness;
- 30% visibility percentile;
- 10% position-volatility percentile.

## One reason code

`visible_below_expected_ctr`

## One action label

`review_title_meta_and_intent`

The action means human review, not automatic editing.

In [23]:
audit_frame["visibility_bucket"] = pd.cut(
    audit_frame["feature_impressions"],
    bins=[
        99,
        499,
        1_999,
        9_999,
        np.inf,
    ],
    labels=[
        "100-499",
        "500-1,999",
        "2,000-9,999",
        "10,000+",
    ],
    include_lowest=True,
)

visibility_check = (
    audit_frame
    .groupby(
        "visibility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("feature_impressions", "median"),
    )
    .reset_index()
)

visibility_check["decline_rate_pct"] = (
    100 * visibility_check["decline_rate"]
)

visibility_check

,visibility_bucket,n,declining_pages,decline_rate,median_impressions,decline_rate_pct
0,100-499,34590,12062,0.348714,224.0,34.871350
1,"500-1,999",26261,8784,0.334488,922.0,33.448840
2,"2,000-9,999",13474,4670,0.346593,3443.5,34.659344
3,"10,000+",1876,748,0.398721,14786.5,39.872068


#Signal Check 1 — Visibility — Code Cell

In [24]:
audit_frame["visibility_bucket"] = pd.cut(
    audit_frame["feature_impressions"],
    bins=[
        99,
        499,
        1_999,
        9_999,
        np.inf,
    ],
    labels=[
        "100-499",
        "500-1,999",
        "2,000-9,999",
        "10,000+",
    ],
    include_lowest=True,
)

visibility_check = (
    audit_frame
    .groupby(
        "visibility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("feature_impressions", "median"),
    )
    .reset_index()
)

visibility_check["decline_rate_pct"] = (
    100 * visibility_check["decline_rate"]
)

visibility_check

,visibility_bucket,n,declining_pages,decline_rate,median_impressions,decline_rate_pct
0,100-499,34590,12062,0.348714,224.0,34.871350
1,"500-1,999",26261,8784,0.334488,922.0,33.448840
2,"2,000-9,999",13474,4670,0.346593,3443.5,34.659344
3,"10,000+",1876,748,0.398721,14786.5,39.872068


### Visibility Verdict

**Verdict: MIXED**

The observed decline rate did not increase consistently across all visibility buckets.

The highest-volume bucket had the highest observed decline rate, but the lower and middle buckets were uneven.

Visibility is therefore used as potential impact, not as stand-alone evidence that a page requires a refresh.

#Signal Check 2 — CTR Relative to Position — Code Cell

In [25]:
audit_ctr = audit_frame[
    (audit_frame["feature_impressions"] >= 500)
    & (audit_frame["feature_avg_position"] > 0)
    & (audit_frame["feature_avg_position"] <= 20)
    & (audit_frame["feature_ctr"].notna())
].copy()

audit_ctr["position_band"] = pd.cut(
    audit_ctr["feature_avg_position"],
    bins=[0, 3, 10, 20],
    labels=[
        "Top 3",
        "Page 1",
        "Page 2",
    ],
    include_lowest=True,
)

audit_ctr_reference = (
    audit_ctr
    .groupby(
        "position_band",
        observed=True,
    )["feature_ctr"]
    .median()
    .rename("expected_ctr_for_band")
)

audit_ctr = audit_ctr.merge(
    audit_ctr_reference,
    left_on="position_band",
    right_index=True,
    how="left",
)

audit_ctr["ctr_gap"] = (
    audit_ctr["feature_ctr"]
    - audit_ctr["expected_ctr_for_band"]
)

audit_ctr["ctr_status"] = np.where(
    audit_ctr["ctr_gap"] < 0,
    "Below band median",
    "At or above band median",
)

ctr_signal_check = (
    audit_ctr
    .groupby(
        [
            "position_band",
            "ctr_status",
        ],
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr=("feature_ctr", "median"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

ctr_signal_check["decline_rate_pct"] = (
    100 * ctr_signal_check["decline_rate"]
)

ctr_signal_check

,position_band,ctr_status,n,declining_pages,decline_rate,median_ctr,median_ctr_gap,decline_rate_pct
0,Top 3,At or above band median,3119,577,0.184995,0.005789,0.002778,18.499519
1,Top 3,Below band median,3118,1158,0.371392,0.001378,-0.001633,37.139192
2,Page 1,At or above band median,10489,2512,0.239489,0.004399,0.002284,23.948899
3,Page 1,Below band median,10488,4389,0.418478,0.000940,-0.001175,41.847826
4,Page 2,At or above band median,3413,657,0.192499,0.004673,0.002162,19.249927
5,Page 2,Below band median,3411,1224,0.358839,0.000830,-0.001680,35.883905


#Overall CTR Signal

In [26]:
ctr_signal_summary = (
    audit_ctr
    .groupby(
        "ctr_status",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

ctr_signal_summary["decline_rate_pct"] = (
    100 * ctr_signal_summary["decline_rate"]
)

ctr_signal_summary

,ctr_status,n,declining_pages,decline_rate,median_ctr_gap,decline_rate_pct
0,At or above band median,17021,3746,0.220081,0.002218,22.008108
1,Below band median,17017,6771,0.397896,-0.001350,39.789622


### CTR-Relative-to-Position Verdict

**Verdict: CONFIRMED**

Pages with CTR below the median for comparable position bands showed a meaningfully higher observed decline rate.

This supports using position-adjusted low CTR as the main baseline signal.

The result remains observational and does not prove that editing metadata or content would cause recovery.

# 2. Build the Ranked Queue

The ranked queue uses only feature-window information.

## Operational population

A page must have:

- at least 500 feature-window impressions;
- at least 5 available feature-window days;
- an average position greater than 0 and no more than 20;
- valid feature-window CTR.

## Rule eligibility

A page enters the queue only when its CTR is below the median CTR of its position band.

## Score components

### CTR weakness score

Measures how far the page's CTR falls below the median CTR of comparable pages.

### Visibility score

Uses percentile rank instead of raw impressions because impressions are heavy-tailed.

### Volatility score

Uses percentile rank and receives a small weight because position volatility had a MIXED audit verdict.

In [27]:
operational = baseline_frame.copy()

operational["position_band"] = pd.cut(
    operational["feature_avg_position"],
    bins=[0, 3, 10, 20],
    labels=[
        "Top 3",
        "Page 1",
        "Page 2",
    ],
    include_lowest=True,
)

baseline_ctr_reference = (
    operational
    .groupby(
        "position_band",
        observed=True,
    )["feature_ctr"]
    .median()
    .rename("expected_ctr_for_band")
)

operational = operational.merge(
    baseline_ctr_reference,
    left_on="position_band",
    right_index=True,
    how="left",
)

operational["ctr_gap"] = (
    operational["feature_ctr"]
    - operational["expected_ctr_for_band"]
)

operational["ctr_status"] = np.where(
    operational["ctr_gap"] < 0,
    "Below band median",
    "At or above band median",
)

print(
    operational["ctr_status"].value_counts()
)

assert operational["expected_ctr_for_band"].notna().all()

print("Position-adjusted CTR reference created.")

ctr_status
At or above band median    17040
Below band median          17038
Name: count, dtype: int64
Position-adjusted CTR reference created.


In [28]:
# Percentiles are calculated across the complete operational population,
# before the below-CTR rule is applied.
operational["visibility_score"] = (
    operational["feature_impressions"]
    .rank(
        method="average",
        pct=True,
    )
)

operational["volatility_score"] = (
    operational["feature_position_volatility"]
    .rank(
        method="average",
        pct=True,
    )
)

print("Population-level percentile scores created.")

Population-level percentile scores created.


In [29]:
# Keep only pages that actually satisfy the reason code.
queue = operational[
    operational["ctr_gap"] < 0
].copy()

print(
    "Eligible below-expected-CTR pages:",
    f"{len(queue):,}",
)

assert len(queue) > 0
assert (queue["ctr_gap"] < 0).all()
assert (
    queue["ctr_status"]
    == "Below band median"
).all()

print("Queue rule eligibility passed.")

Eligible below-expected-CTR pages: 17,038
Queue rule eligibility passed.


In [30]:
# Relative CTR weakness
queue["ctr_weakness_score"] = np.where(
    queue["expected_ctr_for_band"] > 0,
    (
        -queue["ctr_gap"]
        / queue["expected_ctr_for_band"]
    ),
    0.0,
)

queue["ctr_weakness_score"] = (
    queue["ctr_weakness_score"]
    .clip(lower=0, upper=1)
)

# Transparent baseline score
queue["baseline_action_score"] = (
    100
    * (
        0.60 * queue["ctr_weakness_score"]
        + 0.30 * queue["visibility_score"]
        + 0.10 * queue["volatility_score"]
    )
)

# Exactly one reason code
queue["reason_code"] = (
    "visible_below_expected_ctr"
)

# Exactly one action label
queue["action_label"] = (
    "review_title_meta_and_intent"
)

# Human-readable confidence note
queue["confidence_note"] = np.select(
    [
        (
            queue["ctr_weakness_score"] >= 0.75
        )
        & (
            queue["visibility_score"] >= 0.75
        ),
        queue["ctr_weakness_score"] >= 0.50,
    ],
    [
        "higher-confidence review candidate",
        "medium-confidence review candidate",
    ],
    default="lower-confidence review candidate",
)

# Rank queue
queue = (
    queue
    .sort_values(
        [
            "baseline_action_score",
            "feature_impressions",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(
    1,
    len(queue) + 1,
)

print(
    "Ranked queue rows:",
    f"{len(queue):,}",
)

queue.head()

Ranked queue rows: 17,038


,client_hash_id,content_hash_id,feature_impressions,feature_clicks,feature_avg_position,feature_position_volatility,feature_available_days,outcome_impressions,outcome_available_days,feature_ctr,...,ctr_gap,ctr_status,visibility_score,volatility_score,ctr_weakness_score,baseline_action_score,reason_code,action_label,confidence_note,rank
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,0.0,8.607910,12.407646,15,62.0,16,0.000000,...,-0.002115,Below band median,0.999795,0.994014,1.000000,99.933975,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,1
1,client_62f4a7e64f5e0096,content_1ff6231687184dec,12634.0,0.0,16.675512,12.485486,15,2246.0,16,0.000000,...,-0.002510,Below band median,0.975703,0.994483,1.000000,99.215916,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,2
2,client_62f4a7e64f5e0096,content_9e0a8a913953b8d3,12020.0,0.0,8.453676,9.402399,14,981.0,15,0.000000,...,-0.002115,Below band median,0.973150,0.983186,1.000000,99.026351,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,3
3,client_fef1a8f436438636,content_551a522809e0358c,7379.0,0.0,13.612519,8.968112,15,3976.0,16,0.000000,...,-0.002510,Below band median,0.936264,0.979870,1.000000,97.886613,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,4
4,client_20259bd6705d81d4,content_1c8b7d9f25c118ab,16368.0,1.0,13.709873,5.866464,15,7321.0,16,0.000061,...,-0.002449,Below band median,0.985592,0.924761,0.975658,97.354849,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,5


In [31]:
queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "confidence_note",
    "position_band",
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "expected_ctr_for_band",
    "ctr_gap",
    "ctr_weakness_score",
    "feature_avg_position",
    "feature_position_volatility",
    "visibility_score",
    "volatility_score",
]

ranked_queue = queue[
    queue_columns
].copy()

ranked_queue.head(10)

,rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action_label,confidence_note,position_band,feature_impressions,feature_clicks,feature_ctr,expected_ctr_for_band,ctr_gap,ctr_weakness_score,feature_avg_position,feature_position_volatility,visibility_score,volatility_score
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,99.933975,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,83772.0,0.0,0.000000,0.002115,-0.002115,1.000000,8.607910,12.407646,0.999795,0.994014
1,2,client_62f4a7e64f5e0096,content_1ff6231687184dec,99.215916,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,12634.0,0.0,0.000000,0.002510,-0.002510,1.000000,16.675512,12.485486,0.975703,0.994483
2,3,client_62f4a7e64f5e0096,content_9e0a8a913953b8d3,99.026351,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,12020.0,0.0,0.000000,0.002115,-0.002115,1.000000,8.453676,9.402399,0.973150,0.983186
3,4,client_fef1a8f436438636,content_551a522809e0358c,97.886613,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,7379.0,0.0,0.000000,0.002510,-0.002510,1.000000,13.612519,8.968112,0.936264,0.979870
4,5,client_20259bd6705d81d4,content_1c8b7d9f25c118ab,97.354849,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,16368.0,1.0,0.000061,0.002510,-0.002449,0.975658,13.709873,5.866464,0.985592,0.924761
5,6,client_20259bd6705d81d4,content_dd76ae989a61d4f6,97.328775,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,6054.0,0.0,0.000000,0.002510,-0.002510,1.000000,19.481478,12.387243,0.912964,0.993984
6,7,client_fef1a8f436438636,content_cc9732f0da1d8d2d,97.106491,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,5702.0,0.0,0.000000,0.002510,-0.002510,1.000000,17.375762,14.309322,0.904587,0.996889
7,8,client_73cda7b4e4f265ea,content_c9f840183215651b,97.035918,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,18421.0,0.0,0.000000,0.002115,-0.002115,1.000000,5.216858,3.244500,0.988878,0.736956
8,9,client_fef1a8f436438636,content_c9643f42e0fb5214,96.289751,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 1,11060.0,1.0,0.000090,0.002115,-0.002024,0.957246,8.800659,9.040149,0.968367,0.980398
9,10,client_20259bd6705d81d4,content_59a897a6ca5418df,95.692969,visible_below_expected_ctr,review_title_meta_and_intent,higher-confidence review candidate,Page 2,4504.0,0.0,0.000000,0.002510,-0.002510,1.000000,16.234793,7.569551,0.868317,0.964346


In [32]:
assert (ranked_queue["ctr_gap"] < 0).all()

assert (
    ranked_queue["reason_code"]
    == "visible_below_expected_ctr"
).all()

assert (
    ranked_queue["action_label"]
    == "review_title_meta_and_intent"
).all()

assert (
    ranked_queue["feature_impressions"] >= 500
).all()

assert (
    ranked_queue["feature_avg_position"] > 0
).all()

assert (
    ranked_queue["feature_avg_position"] <= 20
).all()

print("Reason-code consistency check passed.")
print("Every exported row satisfies the baseline rule.")

Reason-code consistency check passed.
Every exported row satisfies the baseline rule.


In [33]:
ranked_queue.to_csv(
    CSV_PATH,
    index=False,
)

print(
    "Baseline queue written to:",
    CSV_PATH,
)

print(
    "CSV rows:",
    f"{len(ranked_queue):,}",
)

Baseline queue written to: work/outputs/baseline_action_score.csv
CSV rows: 17,038


In [34]:
top_20 = ranked_queue.head(20).copy()

metrics = {
    "assignment": "ML-07",
    "lane": "Refresh / Content Opportunity Scoring",
    "development_month": "2026-03",
    "feature_window": "2026-03-01 to 2026-03-15",
    "queue_rows": int(len(ranked_queue)),
    "top_20_mean_score": float(
        top_20["baseline_action_score"].mean()
    ),
    "top_20_median_impressions": float(
        top_20["feature_impressions"].median()
    ),
    "top_20_mean_ctr_weakness": float(
        top_20["ctr_weakness_score"].mean()
    ),
    "score_weights": {
        "ctr_weakness": 0.60,
        "visibility": 0.30,
        "position_volatility": 0.10,
    },
    "eligibility": {
        "minimum_impressions": 500,
        "minimum_feature_days": 5,
        "minimum_position_exclusive": 0,
        "maximum_position_inclusive": 20,
        "requires_below_position_band_median_ctr": True,
    },
    "reason_code": "visible_below_expected_ctr",
    "action_label": "review_title_meta_and_intent",
}

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        indent=2,
    )

print(
    "Metrics receipt written to:",
    METRICS_PATH,
)

Metrics receipt written to: work/outputs/baseline_metrics.json


# 3. Top-20 Review

I manually review the top 20 as recommendations rather than accepting the score automatically.

For each row, the review includes:

- action;
- reason code;
- confidence note;
- why the page appears in the queue;
- what could make the recommendation wrong.

The review uses pseudonymized identifiers only and exposes no client names, URLs, titles, or private queries.

In [35]:
top_20_review = ranked_queue.head(20).copy()

if len(top_20_review) < 20:
    raise ValueError(
        "The queue contains fewer than 20 rows."
    )

volatility_p90 = (
    ranked_queue[
        "feature_position_volatility"
    ].quantile(0.90)
)


def build_why_note(row):
    return (
        f"{int(row['feature_impressions']):,} impressions, "
        f"average position {row['feature_avg_position']:.1f}, "
        f"CTR {row['feature_ctr']:.4f} versus "
        f"position-band median "
        f"{row['expected_ctr_for_band']:.4f}, "
        f"CTR weakness {row['ctr_weakness_score']:.2f}."
    )


def build_wrong_note(row):
    reasons = []

    if (
        row["feature_position_volatility"]
        >= volatility_p90
    ):
        reasons.append(
            "temporary ranking volatility"
        )

    if row["feature_ctr"] == 0:
        reasons.append(
            "zero clicks may reflect query mix or SERP behaviour"
        )

    if row["feature_avg_position"] > 15:
        reasons.append(
            "low CTR may be partly expected near the bottom of page two"
        )

    if len(reasons) < 3:
        reasons.append("seasonality")

    if len(reasons) < 3:
        reasons.append(
            "search intent may not be fixable through metadata"
        )

    if len(reasons) < 3:
        reasons.append(
            "SERP features may reduce click opportunity"
        )

    return "; ".join(reasons[:3])


top_20_review["why_selected"] = (
    top_20_review.apply(
        build_why_note,
        axis=1,
    )
)

top_20_review["what_would_make_it_wrong"] = (
    top_20_review.apply(
        build_wrong_note,
        axis=1,
    )
)

top_20_review[
    [
        "rank",
        "action_label",
        "reason_code",
        "confidence_note",
        "why_selected",
        "what_would_make_it_wrong",
    ]
]

,rank,action_label,reason_code,confidence_note,why_selected,what_would_make_it_wrong
0,1,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"83,772 impressions, average position 8.6, CTR ...",temporary ranking volatility; zero clicks may ...
1,2,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"12,634 impressions, average position 16.7, CTR...",temporary ranking volatility; zero clicks may ...
2,3,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"12,020 impressions, average position 8.5, CTR ...",temporary ranking volatility; zero clicks may ...
3,4,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"7,379 impressions, average position 13.6, CTR ...",temporary ranking volatility; zero clicks may ...
4,5,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"16,368 impressions, average position 13.7, CTR...",temporary ranking volatility; seasonality; sea...
5,6,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"6,054 impressions, average position 19.5, CTR ...",temporary ranking volatility; zero clicks may ...
6,7,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"5,702 impressions, average position 17.4, CTR ...",temporary ranking volatility; zero clicks may ...
7,8,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"18,421 impressions, average position 5.2, CTR ...",zero clicks may reflect query mix or SERP beha...
8,9,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"11,060 impressions, average position 8.8, CTR ...",temporary ranking volatility; seasonality; sea...
9,10,review_title_meta_and_intent,visible_below_expected_ctr,higher-confidence review candidate,"4,504 impressions, average position 16.2, CTR ...",temporary ranking volatility; zero clicks may ...


In [14]:
for _, row in top_20_review.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"Action={row['action_label']} | "
        f"Reason={row['reason_code']} | "
        f"Confidence={row['confidence_note']} | "
        f"Why={row['why_selected']} | "
        f"Could be wrong if={row['what_would_make_it_wrong']}"
    )

Rank 1: Action=review_title_meta_and_intent | Reason=visible_below_expected_ctr | Confidence=higher-confidence review candidate | Why=83,772 impressions, position 8.6, CTR 0.0000 versus band median 0.0021, with CTR weakness score 1.00. | Could be wrong if=temporary ranking volatility; zero clicks may reflect query mix or SERP behaviour; seasonality
Rank 2: Action=review_title_meta_and_intent | Reason=visible_below_expected_ctr | Confidence=higher-confidence review candidate | Why=12,634 impressions, position 16.7, CTR 0.0000 versus band median 0.0025, with CTR weakness score 1.00. | Could be wrong if=temporary ranking volatility; zero clicks may reflect query mix or SERP behaviour; low CTR may be normal near the bottom of page two
Rank 3: Action=review_title_meta_and_intent | Reason=visible_below_expected_ctr | Confidence=higher-confidence review candidate | Why=12,020 impressions, position 8.5, CTR 0.0000 versus band median 0.0021, with CTR weakness score 1.00. | Could be wrong if=tem

## Top-20 Review Observation

The top-ranked pages combine substantial visibility with CTR below the median for comparable position bands.

They are reasonable review candidates because:

- their search exposure is large enough to matter;
- their CTR weakness is measured relative to similar ranking positions;
- their score is transparent and reproducible.

However, every recommendation can still be wrong. Low CTR may reflect seasonality, query mix, search intent, SERP features, brand recognition, or measurement noise.

The action is therefore review, not automatic editing.

# 4. Weak Picks and Leakage Check

A weak pick is a page where:

- the CTR weakness score is below 0.75; or
- the average position is greater than 15, where lower CTR may partly be expected.

These picks are not automatically incorrect. They simply require more skepticism during human review.

In [36]:
top_20_review["weak_pick"] = (
    (
        top_20_review[
            "ctr_weakness_score"
        ] < 0.75
    )
    | (
        top_20_review[
            "feature_avg_position"
        ] > 15
    )
)

top_20_review["weak_pick_reason"] = np.select(
    [
        (
            top_20_review[
                "ctr_weakness_score"
            ] < 0.75
        ),
        (
            top_20_review[
                "feature_avg_position"
            ] > 15
        ),
    ],
    [
        "CTR weakness is not especially strong",
        (
            "Low CTR may be partly expected "
            "near the bottom of page two"
        ),
    ],
    default="No obvious weak-pick condition",
)

weak_picks = top_20_review[
    top_20_review["weak_pick"]
][
    [
        "rank",
        "baseline_action_score",
        "feature_impressions",
        "feature_avg_position",
        "ctr_weakness_score",
        "visibility_score",
        "volatility_score",
        "weak_pick_reason",
    ]
]

print(
    "Weak picks among top 20:",
    len(weak_picks),
)

weak_picks

Weak picks among top 20: 6


,rank,baseline_action_score,feature_impressions,feature_avg_position,ctr_weakness_score,visibility_score,volatility_score,weak_pick_reason
1,2,99.215916,12634.0,16.675512,1.0,0.975703,0.994483,Low CTR may be partly expected near the bottom...
5,6,97.328775,6054.0,19.481478,1.0,0.912964,0.993984,Low CTR may be partly expected near the bottom...
6,7,97.106491,5702.0,17.375762,1.0,0.904587,0.996889,Low CTR may be partly expected near the bottom...
9,10,95.692969,4504.0,16.234793,1.0,0.868317,0.964346,Low CTR may be partly expected near the bottom...
12,13,95.341129,4046.0,19.616333,1.0,0.848275,0.989289,Low CTR may be partly expected near the bottom...
13,14,95.310171,4417.0,18.135571,1.0,0.865221,0.935354,Low CTR may be partly expected near the bottom...


### Weak-Pick Interpretation

Weak picks are not necessarily wrong.

They are recommendations where position context or a less extreme CTR gap creates additional uncertainty.

A reviewer should examine these pages carefully before recommending changes.

## Leakage and Privacy Check

The score must contain no:

- outcome-window measurements;
- declining labels;
- decline ratios;
- product scores;
- product flags;
- client names;
- URLs;
- domains;
- titles;
- raw private queries.

Outcome measurements were used only in the two signal checks. They were not used in baseline eligibility, scoring, reason codes, actions, confidence labels, or exported queue columns.

In [37]:
baseline_input_columns = {
    "feature_impressions",
    "feature_ctr",
    "expected_ctr_for_band",
    "ctr_gap",
    "feature_avg_position",
    "feature_position_volatility",
    "ctr_weakness_score",
    "visibility_score",
    "volatility_score",
}

forbidden_fields = {
    "outcome_impressions",
    "outcome_daily_impressions",
    "outcome_available_days",
    "is_declining_proxy",
    "decline_ratio",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "refresh_flag",
}

privacy_patterns = {
    "url",
    "domain",
    "title",
    "raw_query",
    "query_text",
    "client_name",
    "email",
    "token",
}

leakage_overlap = sorted(
    baseline_input_columns.intersection(
        forbidden_fields
    )
)

exported_forbidden = sorted(
    set(ranked_queue.columns).intersection(
        forbidden_fields
    )
)

exported_privacy_risks = sorted(
    column
    for column in ranked_queue.columns
    if any(
        pattern in column.lower()
        for pattern in privacy_patterns
    )
)

print(
    "Forbidden score inputs:",
    leakage_overlap,
)

print(
    "Forbidden exported columns:",
    exported_forbidden,
)

print(
    "Privacy-risk exported columns:",
    exported_privacy_risks,
)

assert len(leakage_overlap) == 0
assert len(exported_forbidden) == 0
assert len(exported_privacy_risks) == 0

print("Leakage and privacy checks passed.")

Forbidden score inputs: []
Forbidden exported columns: []
Privacy-risk exported columns: []
Leakage and privacy checks passed.


In [38]:
assert len(ranked_queue) > 0

assert len(top_20_review) == 20

assert ranked_queue[
    "rank"
].is_monotonic_increasing

assert ranked_queue[
    "baseline_action_score"
].is_monotonic_decreasing

assert (
    ranked_queue["ctr_gap"] < 0
).all()

assert (
    ranked_queue["reason_code"]
    == "visible_below_expected_ctr"
).all()

assert (
    ranked_queue["action_label"]
    == "review_title_meta_and_intent"
).all()

assert (
    ranked_queue["feature_impressions"]
    >= 500
).all()

assert (
    ranked_queue["feature_avg_position"]
    > 0
).all()

assert (
    ranked_queue["feature_avg_position"]
    <= 20
).all()

assert ranked_queue[
    "reason_code"
].nunique() == 1

assert ranked_queue[
    "action_label"
].nunique() == 1

assert CSV_PATH.exists()

assert METRICS_PATH.exists()

print("All rule-consistency assertions passed.")

All rule-consistency assertions passed.


In [39]:
print("ML-07 NOTEBOOK RECEIPT")
print("-" * 65)

print(
    "Operational population:",
    f"{len(operational):,}",
)

print(
    "Below-expected-CTR queue rows:",
    f"{len(ranked_queue):,}",
)

print(
    "Top-20 mean score:",
    round(
        top_20_review[
            "baseline_action_score"
        ].mean(),
        3,
    ),
)

print(
    "Top-20 median impressions:",
    int(
        top_20_review[
            "feature_impressions"
        ].median()
    ),
)

print(
    "Top-20 mean CTR weakness:",
    round(
        top_20_review[
            "ctr_weakness_score"
        ].mean(),
        3,
    ),
)

print(
    "Weak picks in top 20:",
    int(
        top_20_review[
            "weak_pick"
        ].sum()
    ),
)

print(
    "Reason code:",
    ranked_queue[
        "reason_code"
    ].unique(),
)

print(
    "Action label:",
    ranked_queue[
        "action_label"
    ].unique(),
)

print(
    "CSV exists:",
    CSV_PATH.exists(),
)

print(
    "Metrics JSON exists:",
    METRICS_PATH.exists(),
)

print(
    "Leakage fields found:",
    leakage_overlap,
)

print(
    "Exported forbidden fields:",
    exported_forbidden,
)

print(
    "Privacy-risk fields:",
    exported_privacy_risks,
)

print("\nML-07 checks passed.")

ML-07 NOTEBOOK RECEIPT
-----------------------------------------------------------------
Operational population: 34,078
Below-expected-CTR queue rows: 17,038
Top-20 mean score: 96.465
Top-20 median impressions: 6832
Top-20 mean CTR weakness: 0.997
Weak picks in top 20: 6
Reason code: ['visible_below_expected_ctr']
Action label: ['review_title_meta_and_intent']
CSV exists: True
Metrics JSON exists: True
Leakage fields found: []
Exported forbidden fields: []
Privacy-risk fields: []

ML-07 checks passed.


## Final Results Summary

This notebook creates a transparent baseline review queue using only feature-window measurements.

The main weakness signal is CTR below the median for comparable ranking positions. This signal received a CONFIRMED verdict in ML-06.

Visibility is converted into a percentile-based impact score because raw impressions are heavily skewed.

Position volatility receives a small weight because its signal-audit verdict was MIXED.

The final score uses:

- 60% position-adjusted CTR weakness;
- 30% visibility percentile;
- 10% position-volatility percentile.

A page enters the queue only when it actually has CTR below its position-band median.

Every exported row therefore truthfully receives the single reason code:

`visible_below_expected_ctr`

Every row receives the action label:

`review_title_meta_and_intent`

The top 20 were reviewed with an explanation of why they appeared and what could make each recommendation wrong.

The baseline is intended for human decision support. It does not prove that a page requires editing or that an edit would cause recovery.

# Self-check

- [x] I checked two signals before building the rule.
- [x] Both signal tables show `n`.
- [x] I used one transparent baseline rule.
- [x] Every exported row actually satisfies the rule.
- [x] The queue has one reason code.
- [x] The queue has one action label.
- [x] The ranked queue was written to `work/outputs/baseline_action_score.csv`.
- [x] A metrics JSON receipt was generated.
- [x] I reviewed all top 20 rows.
- [x] Every top-20 row has an action, reason code, confidence note, explanation, and skeptical note.
- [x] I identified possible weak picks.
- [x] Baseline eligibility uses no outcome-window field.
- [x] The score uses no future-window or label-derived input.
- [x] No product flags or private fields were exported.
- [x] The notebook runs from top to bottom without errors.
- [x] My conclusions use observed, measured, directional, and decision-support language.